# 06 — So sánh dev và khóa cấu hình SVM

Năm biến thể dùng cùng 1.564 ảnh dev và cùng lưới bảy SVC. Cấu hình cuối được chọn bằng Macro-F1 chưa làm tròn, rồi MAE VND; nếu vẫn hòa, giữ thứ tự biến thể và thứ tự ứng viên đã khai báo trước.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
PREVIOUS_ARTIFACT = 'hsv12_hog_pca/svm_grid_dev.csv'
# PREVIOUS_NOTEBOOK_OUTPUT: khôi phục toàn bộ kết quả của notebook ngay trước đó.
from pathlib import Path
from shutil import copytree
import os

if Path('/kaggle/working').is_dir():
    notebook_root = Path('/kaggle/input/notebooks')
    attached = sorted(p for p in notebook_root.glob('*/*') if p.is_dir())
    if len(attached) != 1:
        raise FileNotFoundError(
            'Hãy gắn đúng một Notebook Output của bước ngay trước đó. '
            f'Đang thấy: {attached}'
        )
    configs = [p for p in attached[0].rglob('experiment_config.json')
               if p.parent.name == 'inputs' and p.parent.parent.name == 'results_224']
    if len(configs) != 1:
        raise FileNotFoundError(
            'Notebook Output đã gắn phải chứa results_224/inputs/experiment_config.json. '
            f'Tìm thấy: {configs}'
        )
    source = configs[0].parents[1]
    expected = source / PREVIOUS_ARTIFACT
    if not expected.is_file():
        raise FileNotFoundError(
            f'Output không phải bản chạy thành công của bước ngay trước: thiếu {expected}'
        )
    target = Path(os.environ.get('CS231_SVM_RESULTS_ROOT',
                                 '/kaggle/working/CS231_SVM/results_224'))
    copytree(source, target, dirs_exist_ok=True)
    for relative in ('inputs/train_manifest.csv', 'inputs/dev_manifest.csv'):
        if not (target / relative).is_file():
            raise FileNotFoundError(f'Output thiếu {target / relative}')
    print('Đã chép:', source, '→', target)
else:
    print('Chạy local: dùng results_224 hiện có trong repository.')


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Tổng hợp 35 cấu hình dev

Bảng đầy đủ giữ cả kết quả kém hơn để phạm vi khảo sát minh bạch. Đây là 35 tổ hợp trên một split dev, không phải 35 lần lặp độc lập.


In [ ]:
from src.SVM.pipeline import FEATURE_ORDER, candidates, sha256
SELECTION = RESULTS_ROOT / 'selection'
SELECTION.mkdir(parents=True, exist_ok=True)
grids = []
source_files = []
for feature_order, feature_id in enumerate(FEATURE_ORDER):
    folder = RESULTS_ROOT / FEATURE_FOLDERS[feature_id]
    path = folder / 'svm_grid_dev.csv'
    grid = pd.read_csv(path)
    if len(grid) != len(candidates()) or set(grid['order']) != set(range(len(candidates()))):
        raise ValueError(f'Lưới SVM không đủ cấu hình: {path}')
    grid['feature_id'] = feature_id
    grid['feature_order'] = feature_order
    grids.append(grid)
    source_files.append(path)
all_results = pd.concat(grids, ignore_index=True).sort_values(
    ['macro_f1', 'mae_vnd', 'feature_order', 'order'],
    ascending=[False, True, True, True]
).reset_index(drop=True)
all_results.to_csv(SELECTION / 'all_35_configurations_dev.csv', index=False)
print('Tổng cấu hình:', len(all_results))
display(all_results.head(10))


## 2. Bảng tốt nhất và đối chứng cố định

Bảng tốt nhất cho biết tổ hợp biểu diễn và SVC thắng trong từng nhánh. Bảng `linear, C=1` giữ bộ phân loại cố định để đọc tác động của biểu diễn rõ hơn.


In [ ]:
columns = ['feature_id', 'kernel', 'C', 'gamma', 'accuracy', 'macro_f1', 'mae_vnd']
best_by_feature = all_results.drop_duplicates('feature_id', keep='first')
reference = all_results.loc[(all_results.kernel == 'linear') & (all_results.C == 1.0)]
reference = reference.sort_values('feature_order')
if len(best_by_feature) != 5 or len(reference) != 5:
    raise ValueError('Thiếu một biến thể hoặc cấu hình tham chiếu')
best_by_feature[columns].to_csv(SELECTION / 'best_by_feature_dev.csv', index=False)
reference[columns].to_csv(SELECTION / 'reference_dev.csv', index=False)
display(best_by_feature[columns])
display(reference[columns])


## 3. Ghi khóa và hash đầu vào

Khóa ghi lại biểu diễn, tham số SVC, thứ tự lớp, ma trận feature train/dev, manifest, grid và mã xử lý. SHA-256 phát hiện thay đổi file; nó không chứng minh các ảnh độc lập hoặc nhãn chính xác.


In [ ]:
from datetime import datetime, timezone
import platform, sklearn, cv2
best = all_results.iloc[0]
feature_id = str(best.feature_id)
folder = RESULTS_ROOT / FEATURE_FOLDERS[feature_id]
X_train, y_train, train_manifest = load_features(folder, 'train')
X_dev, y_dev, dev_manifest = load_features(folder, 'dev')
protocol_file = RESULTS_ROOT / 'inputs/experiment_config.json'
protocol = json.loads(protocol_file.read_text(encoding='utf-8'))
gamma = best.gamma if best.gamma == 'scale' else float(best.gamma)
params = {'kernel': str(best.kernel), 'C': float(best.C), 'gamma': gamma,
          'cache_size': 512, 'decision_function_shape': 'ovr', 'break_ties': True}
result_paths = source_files + [protocol_file]
result_paths += [folder / name for name in
                 ['X_train.npy', 'X_dev.npy', 'y_train.npy', 'y_dev.npy',
                  'train_manifest.csv', 'dev_manifest.csv']]
if 'PCA' in feature_id:
    result_paths.append(RESULTS_ROOT / 'hog_pca/pca.pkl')
project_paths = [PROJECT_ROOT / name for name in
                 ['src/SVM/pipeline.py', 'src/preprocessing.py',
                  'src/evaluation/value_metrics.py']]
hashes = {f'results/{path.relative_to(RESULTS_ROOT).as_posix()}': sha256(path)
          for path in result_paths}
hashes.update({f'project/{path.relative_to(PROJECT_ROOT).as_posix()}': sha256(path)
               for path in project_paths})
lock = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'feature_id': feature_id, 'feature_folder': FEATURE_FOLDERS[feature_id],
    'feature_dim': int(X_train.shape[1]), 'feature_dtype': str(X_train.dtype),
    'classes_vnd': CLASSES, 'svm': params,
    'dev_metrics': {'accuracy': float(best.accuracy),
                    'macro_f1': float(best.macro_f1), 'mae_vnd': float(best.mae_vnd)},
    'training_split': 'train', 'refit_train_plus_dev': False,
    'grid_configurations': len(all_results),
    'selection_rule': 'max unrounded macro_f1; min unrounded mae_vnd; feature order; candidate order',
    'experiment_protocol': protocol, 'sha256': hashes,
    'test_history': 'Previously inspected; not an untouched holdout.',
    'environment_at_lock': {'python': platform.python_version(),
                            'sklearn': sklearn.__version__, 'opencv': cv2.__version__,
                            'numpy': np.__version__, 'pandas': pd.__version__},
}
lock_path = SELECTION / 'locked_svm_config.json'
if lock_path.exists():
    old = json.loads(lock_path.read_text(encoding='utf-8'))
    if old['sha256'] != hashes or old['feature_id'] != feature_id or old['svm'] != params:
        raise ValueError('Cấu hình đã khóa khác đầu vào hiện tại; tạo lượt thực nghiệm mới trước khi chấm test')
    print('Giữ nguyên cấu hình đã khóa:', lock_path)
else:
    save_json(lock_path, lock)
    print('Đã khóa:', feature_id, params)
